<a href="https://colab.research.google.com/github/gitsish/Coding-Motivation/blob/main/Copy_of_FineTuning_0409.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
2+2

4

In [ ]:
2+8

10

In [ ]:
# ================================
# 1. Install Dependencies
# ================================
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# Install Unsloth + extra deps
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major_version >= 8:
    !pip install -q --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes

!pip install -q evaluate rouge_score wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
# ================================
# 2. Import Libraries
# ================================
from unsloth import FastLanguageModel
from trl import SFTTrainer, DPOTrainer, DPOConfig
from transformers import TrainingArguments, AutoTokenizer
from datasets import load_dataset, Dataset, DatasetDict
import evaluate
import wandb
import pandas as pd
from sklearn.model_selection import train_test_split

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# ================================
# 3. Initialize W&B
# ================================
wandb.login()   # Make sure you paste your W&B API key
wandb.init(
    project="finetuning_0409",
    name="sft_run",
    config={"model": "Qwen2.5-0.5B", "method": "LoRA + DPO"}
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aaishasultana (aaishasultana-prasad-v-potluri-siddhartha-institute-of-t) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# ================================
# 5. Load & Prepare Dataset (SFT)
# ================================
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# Keep small subset for demo
dataset = dataset.shuffle(seed=42).select(range(500))

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 500
})

In [ ]:
dataset[0]

{'output': 'Early, she left the party.',
 'input': 'She left the party early',
 'instruction': 'Rearrange the following sentence to make the sentence more interesting.'}

input+output =max seq tokens

In [ ]:
# ================================
# 4. Load Base Model + Tokenizer
# ================================
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.9.1: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=276,
)

Unsloth 2025.9.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [ ]:
# Original matrix = W ( 10000*10000)
# FInal adaptor = A*B ( r= 8)
# Lets say W1 = A*B
# Final weight = W + (alpha/r) (A*B) = W + (alpha/8)(A*B)  = W + (alpha/8) *W1
# Alpha = 0.000001
# FInal weight =  W + (0.000001/8)*W1 ~ W

# ****************************************************************************
# Alpha = 200
# FInal weight =  W + (200/8)*W1 =  W + 25*W1
# ****************************************************************************
# Alpha = rank → We are giving same weight to adaptor and original matrix
# Alpha = 2*rank

# ### Instead of storing all intermediate activations for backpropagation,
# ### the model will recompute some forward passes during the backward pass.



In [ ]:
# Format prompts
def formatting_prompts_func(examples):
    template = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = template.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

In [ ]:
# # {'output': 'Global warming can be reversed by reducing greenhouse gas emissions and deforestation.',
# #  'input': 'Global warming can be reversed by reducing ________ and __________.',
# #  'instruction': 'Fill in the blanks to complete the sentence.'}


#  text = """Below is an instruction that describes a task, paired with an input that provides further context.
# Write a response that appropriately completes the request.

# ### Instruction:
# {'Fill in the blanks to complete the sentence.'}

# ### Input:
# {Global warming can be reversed by reducing ________ and __________.}

# ### Response:
# {Global warming can be reversed by reducing greenhouse gas emissions and deforestation.}"""

In [ ]:
# Format prompts
def formatting_prompts_func(examples):
    template = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = template.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 500
})

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
dataset[4]['text']

'Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request.\n\n### Instruction:\nFill in the blanks to complete the sentence.\n\n### Input:\nGlobal warming can be reversed by reducing ________ and __________.\n\n### Response:\nGlobal warming can be reversed by reducing greenhouse gas emissions and deforestation.<|endoftext|>'

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset, eval_dataset = dataset["train"], dataset["test"]

In [ ]:
train_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 450
})

In [ ]:
eval_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 50
})

In [ ]:
# ================================
# 6. Define TrainingArguments (SFT)
# ================================
training_args = TrainingArguments(
    do_eval=True,
    eval_steps=100,
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=5,
    learning_rate=2e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=3507,
    output_dir="./sft_model",
    report_to="wandb",    # 🔹 Enable W&B logging
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    max_seq_length=max_seq_length,
    dataset_kwargs={"add_special_tokens": False, "append_concat_token": False},
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 5 | Total steps = 145
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,entropy
5,1.860800,0
10,1.904600,No Log
15,1.799400,No Log
20,1.786100,No Log
25,1.694200,No Log
30,1.778400,No Log
35,1.599200,No Log
40,1.512300,No Log
45,1.434400,No Log
50,1.483600,No Log


TrainOutput(global_step=145, training_loss=1.4483228354618467, metrics={'train_runtime': 288.9402, 'train_samples_per_second': 7.787, 'train_steps_per_second': 0.502, 'total_flos': 1722114602757120.0, 'train_loss': 1.4483228354618467, 'epoch': 5.0})

In [ ]:
trainer.save_model("final_sft_model")
tokenizer.save_pretrained("final_sft_model")

('final_sft_model/tokenizer_config.json',
 'final_sft_model/special_tokens_map.json',
 'final_sft_model/vocab.json',
 'final_sft_model/merges.txt',
 'final_sft_model/added_tokens.json',
 'final_sft_model/tokenizer.json')

In [ ]:

## Download model and tokenizer
## Pass our input to tokenizor

## Pass the tokenozed input to model

## Output of model has to be decoded back

In [ ]:
prompt = "Explain gravity to a 10 year old."
out = model.generate(**tokenizer(prompt, return_tensors="pt").to(model.device), max_new_tokens=100)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Explain gravity to a 10 year old. Gravity is like a strong invisible force that pulls everything down, like a giant tug-of-war. It's like a big invisible hand that pulls everything down, like a big, heavy, heavy thing. It's like a big, heavy, heavy thing that pulls everything down, like a big, heavy, heavy thing. It's like a big, heavy, heavy thing that pulls everything down, like a big, heavy, heavy thing. It's like a big, heavy, heavy thing that pulls


In [ ]:
eval_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 50
})

In [ ]:
def generate_answer(model, tokenizer, prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
eval_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 50
})

In [ ]:
preds, refs = [], []

for row in eval_dataset:
    instr = row["instruction"]
    inp   = row.get("input", "")
    ref   = row["output"].strip()

    # Build prompt in Alpaca-style
    if inp:
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"

    pred = generate_answer(model, tokenizer, prompt)
    preds.append(pred)
    refs.append(ref)

In [ ]:
preds[49]

"### Instruction:\nAdd 3 details to the text to make it more interesting.\n\n### Input:\nThe cat sat on the windowsill.\n\n### Response:\nThe cat sat on the windowsill, its fur swaying gently in the gentle breeze. The windowsill was adorned with a collection of colorful flowers, each one with a unique design and arrangement. The cat, with its fluffy fur, moved from one flower to another, taking in the sights and sounds of the surrounding environment. The windowsill was a cozy and inviting spot for the cat to rest and relax, its warmth and comfort filling the air with a sense of peace and tranquility. The cat's expression was one of contentment as it gazed upon the world around it, its eyes sparkling with excitement and curiosity. The windowsill"

In [ ]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rouge_scores = rouge.compute(predictions=preds, references=refs)
bleu_scores = bleu.compute(predictions=preds, references=refs)

print("ROUGE Scores:", rouge_scores)
print("BLEU Score:", bleu_scores)

ROUGE Scores: {'rouge1': np.float64(0.3246519276977805), 'rouge2': np.float64(0.13896903131819355), 'rougeL': np.float64(0.22675632972840504), 'rougeLsum': np.float64(0.2754147854008153)}
BLEU Score: {'bleu': 0.08890904462425749, 'precisions': [0.34919804741980476, 0.12873724938445305, 0.06210078069552875, 0.03634085213032581], 'brevity_penalty': 0.8858881528921773, 'length_ratio': 0.8919297154408334, 'translation_length': 5736, 'reference_length': 6431}


DPO

In [ ]:
from unsloth import FastLanguageModel
from trl import DPOTrainer
from peft import PeftModel
from transformers import AutoTokenizer, TrainingArguments
from datasets import load_dataset
import torch

max_seq_length = 1024
dtype=None
load_in_4bit = True




In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/final_sft_model",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
import pandas as pd

In [37]:
df = pd.read_csv('dpo_data.csv')
df.shape

(50, 3)

In [38]:
df.head()

,prompt,chosen,rejected
0,Write a short poem about AI.,"AI, a spark of human mind,\nA helper, patient,...",AI is here. AI is smart. It does things. That'...
1,Explain gravity to a 10 year old.,Gravity is like an invisible hand that pulls e...,Gravity is a complex spacetime curvature defin...
2,Summarize the plot of Romeo and Juliet.,Romeo and Juliet are two young lovers from fam...,Romeo and Juliet are a story about two people....
3,List 3 healthy breakfast ideas.,1. Oatmeal with fruits and nuts\n2. Whole-grai...,"Eat donuts, chips, and soda for breakfast."
4,Translate 'Good morning' to French.,The translation of 'Good morning' in French is...,'Good morning' in French is 'Gracias'.


In [39]:
# 🔹 Train-test split (e.g. 90% train, 10% test)
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# 🔹 Convert Pandas DataFrames → Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# 🔹 Wrap into DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 45
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 5
    })
})


In [40]:
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    output_dir="./dpo-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    num_train_epochs=5,
    logging_steps=2,
    save_steps=100,
    # evaluation_strategy="steps",
    save_total_limit=2,
    report_to="none",   # or "wandb" if using wandb
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,               # baseline reference model (can pass your SFT checkpoint if you want)
    args=training_args,
    beta=0.05,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
)

# from unsloth import PatchDPOTrainer
# PatchDPOTrainer()

trainer.train()


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/45 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/45 [00:00<?, ? examples/s]

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Extracting prompt in eval dataset (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

...

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Tokenizing eval dataset (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 45 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
2,0.682500,0.045849,0.023902,0.625000,0.021947,-34.739094,-34.301392,-1.293929,-1.376299,0,0,0,0
4,0.680100,0.035114,0.007906,0.875000,0.027208,-31.261351,-38.502552,-1.908827,-1.583494,No Log,No Log,No Log,No Log
6,0.689700,0.034813,0.026341,0.625000,0.008472,-42.141876,-43.523643,-1.765875,-1.263698,No Log,No Log,No Log,No Log
8,0.689800,0.033074,0.025308,0.875000,0.007766,-27.134829,-33.250412,-1.933797,-1.724078,No Log,No Log,No Log,No Log
10,0.670500,0.034071,-0.012776,0.875000,0.046847,-34.148918,-39.764816,-1.901055,-2.065592,No Log,No Log,No Log,No Log
12,0.691000,0.027024,0.026137,0.666667,0.000887,-33.142399,-32.237392,-2.102339,-1.951781,No Log,No Log,No Log,No Log
14,0.661100,0.042049,-0.024431,0.875000,0.066480,-28.927532,-35.695553,-1.806231,-1.774642,No Log,No Log,No Log,No Log
16,0.652800,0.081966,-0.003040,0.750000,0.085006,-35.503155,-38.663937,-1.811521,-1.619986,No Log,No Log,No Log,No Log
18,0.663700,0.034651,-0.025879,0.875000,0.060530,-31.104378,-38.182407,-1.749377,-1.701609,No Log,No Log,No Log,No Log
20,0.666300,0.038816,-0.016630,1.000000,0.055445,-29.878418,-29.432108,-2.188666,-1.591900,No Log,No Log,No Log,No Log


TrainOutput(global_step=60, training_loss=0.6401096681753794, metrics={'train_runtime': 58.9411, 'train_samples_per_second': 3.817, 'train_steps_per_second': 1.018, 'total_flos': 0.0, 'train_loss': 0.6401096681753794, 'epoch': 5.0})